# 11 — Caching & Persistence

**What you will learn:**
- Why caching matters: lazy evaluation and recomputation
- `cache()` vs `persist()` — the difference
- All Storage Levels with trade-offs
- When to cache and when NOT to cache
- `unpersist()` — releasing cached data
- Verifying cache in Spark UI

## 1. Why Caching Matters

Spark uses **lazy evaluation**. Every action on a DataFrame re-executes the full lineage from scratch.
If you reuse a DataFrame across multiple actions, that lineage runs **once per action** — wasted I/O.

```
Without cache:
  source → filter → join → agg   ← action 1  (reads source)
  source → filter → join → agg   ← action 2  (reads source AGAIN)

With cache:
  source → filter → join  → cache()
                              ↳ action 1  (materializes into memory)
                              ↳ action 2  (served from memory — fast!)
```

In [ ]:
import pyspark.sql.functions as F
from pyspark import StorageLevel
import time

base = [(i, f"emp_{i}", ["Engineering","Marketing","HR","Finance"][i%4],
         50000.0 + (i * 137 % 70000)) for i in range(1, 200001)]

df = spark.createDataFrame(base, ["id","name","department","salary"])
print(f"Total rows: {df.count():,}")

## 2. cache() — Default In-Memory Caching

`cache()` is shorthand for `persist(StorageLevel.MEMORY_AND_DISK_DESER)`.
It is **lazy** — data is NOT written to memory until the first action fires.

In [ ]:
df_expensive = (
    df.filter(F.col("salary") > 80000)
      .withColumn("bonus", F.round(F.col("salary") * 0.15, 2))
      .withColumn("dept_upper", F.upper(F.col("department")))
)

df_expensive.cache()   # marks for caching — nothing stored yet

# First action: materializes the cache
t0 = time.time()
c1 = df_expensive.count()
print(f"1st action (cache MISS — builds cache): {c1:,}  in {time.time()-t0:.2f}s")

# Second action: served from cache
t0 = time.time()
c2 = df_expensive.count()
print(f"2nd action (cache HIT):                 {c2:,}  in {time.time()-t0:.2f}s")

print("Is cached:", df_expensive.is_cached)

## 3. persist() — Explicit Storage Level

`persist()` gives full control over where and how data is stored.

| Storage Level | Memory | Disk | Serialized | Replicated | Best For |
|---|---|---|---|---|---|
| `MEMORY_ONLY` | Yes | No | No | No | Small DFs that fit in RAM |
| `MEMORY_AND_DISK` | Yes | Spill | No | No | General purpose (recommended) |
| `MEMORY_ONLY_SER` | Yes | No | Yes | No | Memory-constrained, smaller footprint |
| `MEMORY_AND_DISK_SER` | Yes | Spill | Yes | No | Large DFs, high memory pressure |
| `DISK_ONLY` | No | Yes | Yes | No | Very large DFs, rarely re-read |
| `MEMORY_AND_DISK_2` | Yes | Spill | No | Yes (×2) | Fault-tolerant iterative jobs |
| `OFF_HEAP` | Off-heap | No | Yes | No | Avoid JVM GC pauses |

In [ ]:
df_eng = df.filter(F.col("department") == "Engineering")

df_eng.persist(StorageLevel.MEMORY_AND_DISK)
df_eng.count()   # trigger materialization
print("Persisted MEMORY_AND_DISK:", df_eng.is_cached)

# Reuse cheaply
df_eng.groupBy("department").agg(F.avg("salary")).show()
df_eng.orderBy(F.col("salary").desc()).limit(5).show()

## 4. When to Cache vs When NOT to Cache

**Cache WHEN:**
- DataFrame is reused in **2+ actions or branches**
- Result of an **expensive computation** (multi-table join, large aggregation)
- Running **iterative algorithms** (ML training loops)
- Source data does **not change** between actions

**Do NOT cache WHEN:**
- DataFrame is used **only once** — overhead with zero benefit
- DataFrame is **larger than cluster memory** — causes eviction and thrashing
- Computation is **trivially cheap** to repeat (simple filter, small table)
- **Streaming** jobs — continuous micro-batches invalidate caches

In [ ]:
# Good pattern: cache a shared base, branch into two aggregations
df_base = (
    df.join(
        df.groupBy("department").agg(F.avg("salary").alias("dept_avg")),
        on="department", how="left"
    )
    .withColumn("above_avg", F.col("salary") > F.col("dept_avg"))
)
df_base.cache()
df_base.count()   # materialize

# Two branches share the same cached base — no recomputation
above = df_base.filter(F.col("above_avg") == True).groupBy("department").count()
below = df_base.filter(F.col("above_avg") == False).groupBy("department").count()
above.show()
below.show()

## 5. unpersist() — Always Release When Done

Cached DataFrames hold cluster memory until the session ends or you call `unpersist()`.
Not releasing caches causes memory pressure and evicts other useful cached data.

In [ ]:
df_expensive.unpersist()
df_eng.unpersist()
df_base.unpersist()

print("df_expensive cached:", df_expensive.is_cached)
print("df_eng       cached:", df_eng.is_cached)
print("df_base      cached:", df_base.is_cached)

## 6. Verifying Cache in Spark UI

1. Go to **Cluster → Spark UI** in Databricks
2. Click the **Storage** tab
3. Each cached DataFrame is listed with:
   - Storage level
   - Memory consumed
   - Disk spill amount
   - % of partitions materialized (fraction cached)

If fraction cached < 1.0, the DataFrame did not fully fit in memory — consider `MEMORY_AND_DISK_SER` to reduce footprint.

## Key Takeaways

| Concept | Detail |
|---|---|
| `cache()` | Shorthand for `MEMORY_AND_DISK` — lazy, triggered on first action |
| `persist(level)` | Fine-grained: choose memory vs disk vs serialization |
| Best default | `MEMORY_AND_DISK` — spills safely if memory is full |
| Cache when | DataFrame reused 2+ times, expensive lineage |
| Don't cache when | Single use, huge data, cheap recomputation |
| Always | Call `unpersist()` when done — prevents memory leaks |
| Verify | Spark UI → Storage tab |